# UAV human detection across RGB, thermal and paired fusion

This notebook runs the **same ten models** with seeds **17, 42 and 93**, **100 epochs**, input **640**, batch **8** and a common training/evaluation recipe. Use the complete project folder; this notebook imports `uav_review` and reads the bundled weights and images.

**Available data:** 182 thermal pseudocolor images and 216 visible-light RGB images, with all original annotations. These are separate collections. No verified synchronized RGB/thermal pairs were supplied. Their RGB channel storage does not make thermal images visible-light images.

The default run prepares and evaluates RGB and thermal separately. Fusion is marked **AWAITING_VERIFIED_PAIRS**. When actual aligned pairs and common-coordinate labels are supplied in `modalities/fusion/pairs_VERIFIED.csv`, the notebook automatically prepares **all three modalities with the exact same paired cases and partitions**. It never pairs unrelated images by filename or converts one sensor into another.

Start JupyterLab with `Start_JupyterLab_Anaconda.bat`, select **Python (UAV All Modalities)** and run the cells in order. `README.md` contains the Windows/Anaconda setup and the exact fusion-input requirements.

## 1. Locate the complete project and install in this kernel

Leave `ROOT_INPUT` blank when this notebook is opened from the project folder. The installation preserves an existing compatible Torch installation. The supplied NVIDIA setup uses the official CUDA 11.8 wheels; select the CPU index for a CPU-only machine. If already installed in the correct environment, set `INSTALL_DEPENDENCIES = False`.

In [ ]:
import os, sys, json
from pathlib import Path

ROOT_INPUT = r""  # optional full local path to the extracted UAV_Full_Project folder
INSTALL_DEPENDENCIES = True
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu118"  # CPU alternative: https://download.pytorch.org/whl/cpu
CHECK_ONLY = os.environ.get("UAV_CHECK_ONLY", "0") == "1"  # used only for engineering validation

if ROOT_INPUT:
    ROOT = Path(ROOT_INPUT).expanduser().resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    ROOT = next((p for p in candidates if (p / "multimodal_config.json").is_file()
                 and (p / "uav_review/modality_study.py").is_file()), None)
    if ROOT is None:
        raise FileNotFoundError("Extract the full project and set ROOT_INPUT to its UAV_Full_Project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from uav_review.notebook import install_notebook_dependencies
install_notebook_dependencies(ROOT, install=INSTALL_DEPENDENCIES and not CHECK_ONLY,
                              torch_index_url=TORCH_INDEX_URL)
print("Project:", ROOT)
print("Kernel Python:", sys.executable)

## 2. Declare the study before training

All enabled modalities use the same model list and seed list. `DESIGN = "auto"` uses a paired comparison only when the supplied pair manifest passes validation. Otherwise, it runs the two available collections separately and records why fusion is unavailable.

`EVALUATION_MODE = "exploratory"` groups known augmentation families and identical pixels without inventing flight sessions. This permits local runs without the earlier missing-session-file error. Temporal/scene dependence may remain. For verified acquisition metadata, use `"verified_sessions"` or `"grouped_cv"` and the documented session maps. A changed dataset/recipe or moved completed study needs a **new study name**.

In [ ]:
MODALITIES = ["thermal", "rgb", "fusion"]
MODELS = ["yolov5s", "yolov5su", "yolov8n", "yolov8m", "yolov8l", "rtdetr_l",
          "yolov10n", "yolo11n", "yolo12n", "yolo26n"]
SEEDS = [17, 42, 93]
STUDY_NAME = os.environ.get("UAV_STUDY_NAME", "all_modalities_v1")
DEVICE_INPUT = "auto"  # auto, 0, cuda:0, or cpu
EVALUATION_MODE = "exploratory"
DESIGN = "auto"  # paired requires a valid pair manifest; independent keeps the original cohorts separate
FUSION_RGB_WEIGHT = 0.5  # predeclared fixed input-fusion baseline, not a learned feature-fusion module

RUN_TRAINING = not CHECK_ONLY
RUN_EVALUATION = True
RUN_TIMING = True
TRAINING_OVERRIDES = {}  # primary defaults: 100 epochs, 640 pixels, batch 8; changes apply to ALL modalities

OVERRIDES = {"study_name": STUDY_NAME, "modalities": MODALITIES, "models": MODELS, "seeds": SEEDS,
             "device": DEVICE_INPUT, "evaluation_mode": EVALUATION_MODE, "design": DESIGN,
             "fusion": {"rgb_weight": FUSION_RGB_WEIGHT}}
print("Requested modalities:", MODALITIES)
print("Models:", len(MODELS), "| seeds:", SEEDS)
print("Full paired holdout budget:", len(MODALITIES) * len(MODELS) * len(SEEDS), "training jobs")

## 3. Verify Torch, CUDA, native YOLOv5 and the bundled weights

YOLOv5s is the original anchor-based model; YOLOv5su is the separate anchor-free model. Classic YOLOv5 runs in isolated child processes. All ten initial checkpoints are bundled, so training does not need to download weights. CPU execution is available but the full study is intended for your GPU workstation.

In [ ]:
from uav_review.notebook import check_notebook_runtime
from uav_review.engine import MODELS as MODEL_REGISTRY
from uav_review.core import read_json, read_csv, write_json, sha256

RUNTIME = check_notebook_runtime(DEVICE_INPUT)
DEVICE = RUNTIME["selected_device"]
missing = [m for m in MODELS if not (ROOT / "weights" / MODEL_REGISTRY[m]["weights"]).is_file()]
if missing:
    raise FileNotFoundError("Extract the complete project: missing initial weights for " + ", ".join(missing))
for key in MODELS:
    print(f"{key:10s}  {MODEL_REGISTRY[key]['weights']:14s}  {MODEL_REGISTRY[key]['head']}")
print(json.dumps(RUNTIME, indent=2))

## 4. Audit both datasets and prepare the local partitions

This reads every image and annotation, converts valid polygon labels to boxes in a separate working copy, and writes local absolute paths on this machine. Input files are preserved. All models in a modality use the same frozen partitions. In paired mode, all three modalities also share case identities, ground truth and train/validation/test assignments.

The original source-folder counts are distinct from the new exploratory split. No flight-session independence is inferred from filenames.

In [ ]:
import pandas as pd
from IPython.display import display
from uav_review.modality_study import prepare_study

PLAN = prepare_study(ROOT, OVERRIDES)
STUDY = Path(PLAN["study_dir"])
display(pd.DataFrame(PLAN["modality_status"]))
split_counts = []
for modality, cohort in PLAN["cohorts"].items():
    for fold in cohort["folds"]:
        protocol = read_json(Path(fold) / "protocol.json")
        for split, counts in protocol["counts"].items():
            split_counts.append({"modality": modality, "fold": protocol["fold"], "split": split, **counts})
display(pd.DataFrame(split_counts))
print("Study design:", PLAN["design"])
print("Actual scheduled training jobs:", PLAN["scheduled_training_jobs"])
print("Local outputs:", STUDY)

## 5. Inspect actual source examples and annotation counts

These previews are data inspection, not model predictions. Fusion previews appear only when real verified pairs have been provided.

In [ ]:
from uav_review.modality_study import preview_sources
preview_sources(PLAN, per_modality=2)

In [ ]:
import matplotlib.pyplot as plt

table = pd.DataFrame(split_counts)
available = list(PLAN["cohorts"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
colors = {"train": "#173b5e", "val": "#df983c", "test": "#4b9c88"}
for ax, metric, title in zip(axes, ["images", "humans"], ["Prepared images", "Prepared human annotations"]):
    bottom = [0] * len(available)
    for split in ["train", "val", "test"]:
        values = [int(table[(table.modality == m) & (table.split == split)].iloc[0][metric]) for m in available]
        ax.bar(available, values, bottom=bottom, label=split, color=colors[split])
        bottom = [a+b for a,b in zip(bottom, values)]
    ax.set_title(title + " (first fold)")
    ax.legend(frameon=False)
if PLAN["design"] != "paired_modality_comparison":
    fig.suptitle("Separate source collections; no matched modality comparison")
data_figures = STUDY / "data_inspection"
data_figures.mkdir(exist_ok=True)
fig.savefig(data_figures / "prepared_data_counts.png", dpi=220)
plt.show()

## 6. Freeze the common training recipe

Defaults are 100 completed epochs, batch 8, 640-pixel inputs, AdamW and the three declared seeds. Checkpoints use validation mAP@0.5:0.95; confidence uses validation F1. Test data select neither. Early stopping is disabled. The same predeclared augmentations and settings apply to every modality, while native architecture-specific losses are retained.

In [ ]:
from uav_review.modality_study import freeze_training_configs
COMMON_RECIPE = freeze_training_configs(PLAN, device=DEVICE, training_overrides=TRAINING_OVERRIDES)
write_json(STUDY / "notebook_runtime.json", RUNTIME)
print(json.dumps(COMMON_RECIPE, indent=2))
if DEVICE == "cpu" and RUN_TRAINING:
    print("CPU selected: the full 100-epoch study can take substantial time. Your CUDA workstation is recommended.")

## 7. Train, evaluate and measure each model on each ready modality

Each model/seed uses a fresh pretrained initialization in its own child process. Verified completed jobs are reused; interrupted attempts are preserved and restarted as new attempts. A failure is recorded with its stage and other independent jobs continue. Rerun this cell after fixing a failure.

Evaluation reports COCO AP50/AP50:95 and precision/recall/F1 with validation-selected confidence. The numeric `precision` metric and the `numerical_precision = "fp32"` setting use different fields. Seed-17 timing uses batch one, 50 warm-ups and a 60-second loop. Timing includes decode/preprocessing/inference/postprocessing; it excludes capture, communications and tracking. Fusion input generation is offline preprocessing and is not included in model-inference timing.

In [ ]:
from uav_review.modality_study import execute_study

if RUN_TRAINING:
    JOBS = execute_study(PLAN, do_train=True, do_evaluate=RUN_EVALUATION, do_timing=RUN_TIMING,
                         device=DEVICE, training_overrides=TRAINING_OVERRIDES)
    display(pd.DataFrame(JOBS))
else:
    JOBS = []
    print("Training is disabled for this execution. Data preparation is not a completed model experiment.")
    if (STUDY / "job_status.csv").is_file():
        display(pd.read_csv(STUDY / "job_status.csv"))

## 8. Save numerical and visual results from completed evaluations

Reports include completion status, per-seed metrics, model means/sample SDs, learning curves, PR curves, confusion counts, matched qualitative examples and measured runtime records. Missing or failed jobs stay visible. No empirical metric is created for an unavailable modality. Standard deviations describe training variability conditional on the fixed split, not independent-flight uncertainty.

RGB and thermal scores from unmatched collections remain separate descriptive results. They cannot establish a fusion advantage. For a controlled three-modality comparison, provide verified paired inputs and use the shared paired design.

In [ ]:
from uav_review.modality_study import report_study
REPORT = report_study(PLAN)
display(pd.read_csv(REPORT / "completion_status.csv"))
summary = REPORT / "all_modalities_summary.csv"
if summary.is_file():
    display(pd.read_csv(summary))
else:
    print("No completed evaluated models are available in this study; no detection metrics were fabricated.")
print("Results saved in:", REPORT)

In [ ]:
from IPython.display import Image as NotebookImage

for path in sorted(REPORT.rglob("detection_comparison.png")):
    print(path.relative_to(REPORT))
    display(NotebookImage(filename=str(path), width=1000))
for path in sorted(REPORT.rglob("*.csv")):
    print(path.relative_to(REPORT))
print("Per-model learning curves and qualitative figures are inside each modality report folder.")

## 9. Supply the third modality when genuine paired data are available

Put registered visible-light and thermal images in `modalities/fusion/paired_inputs/rgb` and `thermal`, with common-coordinate YOLO labels in `labels`. Complete `modalities/fusion/pairs_VERIFIED.csv` using real acquisition correspondence. The required columns are:

`pair_id,rgb_image,thermal_image,label_path,source_group,paired_verified,alignment_verified,labels_verified_for_both,session_id,provenance_verified`

Paths may be relative to the project folder. Confirm the first three verification fields only after checking real synchronization, pixel registration and label validity for both images. Equal sizes or similar filenames do not prove these properties. `source_group` identifies known related scenes/augmentations; it must not be presented as a flight ID. Verified-session modes additionally require genuine session identifiers and provenance confirmation.

The implemented fusion baseline is `round(alpha * RGB + (1-alpha) * thermal_pseudocolor)` in the shared pixel plane, with fixed `alpha = 0.5` by default. It retains all ten original three-channel model architectures. It is **input-level fusion**, not the manuscript's unvalidated learned/feature-fusion proposal or radiometric fusion. Do not tune alpha on test results.

Select a **new STUDY_NAME**, keep `DESIGN = "auto"` or choose `"paired"`, and rerun from preparation. All three variants are generated from the same matched cases, annotations and shared partitions. Without real pairs, the fusion folders remain empty and its status remains explicit.

## 10. Output locations and reruns

- Inputs: `modalities/thermal`, `modalities/rgb`, `modalities/fusion`.
- Prepared copies, manifests and fold assignments: `experiments/<study_name>/<modality>/audit` and `protocol` (or `paired_inputs` for a paired design).
- Trained checkpoints, histories, predictions and per-run metrics: `experiments/<study_name>/<modality>/training`.
- Job status: `experiments/<study_name>/job_status.csv`.
- Tables and figures: `experiments/<study_name>/reports/report_<timestamp>`.
- Supplied engineering checks: `validation`. One-epoch/128-pixel CPU smoke results are **not** the primary 100-epoch/three-seed study.

Moving an unrun extracted package is safe: preparation creates fresh local paths. After moving a completed study or changing its data/recipe, use a new study name. No manuscript, original source image or original annotation is overwritten.